In [10]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

n = 8                                                  
N = 2**n                                               
n_reduce1 = 4
n_reduce2 = 5
N_reduce1 = 2**n_reduce1 
N_reduce2 = 2**n_reduce2
t = 5                                                   
dt = 0.01                                               
steps = int(t/dt) + 1                                   
x_max = 10                                              
x = np.linspace(-x_max, x_max, N, endpoint=False)      
dx = x[1] - x[0]                                        
                        
freq = np.fft.fftfreq(N, dx)                            
k = 2 * np.pi * freq                                    
k_shift  = np.fft.fftshift(k)                           
L_half = np.exp(-1j * k**2 * dt/4)                      

In [ ]:

def initial_state_soliton_1(x, beta=1/2, xc=0.0, v=2.0):
    profile = 1.0 / np.cosh(beta * (x - xc))


    A = 1.0 / np.linalg.norm(profile)

   
    psi = A * profile * np.exp(1j *0.5 * v * (x - xc))


    g = - beta**2 / A**2
    return psi, g, A

In [ ]:

def initial_state_soliton_2(x, beta=1, xc=0.0, v=2.0):
    profile = 1.0 / np.cosh(beta * (x - xc))


    A = 1.0 / np.linalg.norm(profile)

    psi = A * profile * np.exp(1j *0.5 * v * (x - xc))


    g = - beta**2 / A**2
    return psi, g, A

In [ ]:

def ssfm(psi, g, dt, k, L_half):
    psi = np.fft.ifft(np.fft.fft(psi) * L_half)         
    psi = psi*np.exp(- g*1j * np.abs(psi)**2 * dt)       
    psi = np.fft.ifft(np.fft.fft(psi) * L_half)         

    return psi

In [ ]:

def filtering(modes, N, N_reduce):

    half = N_reduce // 2
    filtered_modes = np.zeros(N , dtype=complex)
    filtered_modes[:half] = modes[:half]          
    filtered_modes[-half:] = modes[-half:]       

    return filtered_modes

In [ ]:

def filtered_qssfm(psi, g, dt, k, L_half, N, N_reduce):
    psi = np.fft.fft(psi) * L_half                                  
    modes = filtering(psi, N, N_reduce)
    psi = np.fft.ifft(psi)

  
    nonlinear_term = np.fft.ifft(modes)
    psi *= np.exp(-g* 1j * np.abs(nonlinear_term)**2 * dt)         

    psi = np.fft.fft(psi) * L_half                                  
    modes = filtering(psi, N, N_reduce)
    psi = np.fft.ifft(psi)
    return psi

In [ ]:

def error(psi, psi_exact):
    rho_exact = np.abs(psi_exact)**2
    rho1 = np.abs(psi)**2
    error = np.sqrt(np.sum((rho1 - rho_exact)**2) / np.sum(rho_exact**2))
    return error

In [ ]:

if __name__ == "__main__":



    psi1, g1, A1 = initial_state_soliton_1(x)
    psi2, g2, A2 = initial_state_soliton_2(x)
    psi1_exact = np.copy(psi1)
    psi2_exact = np.copy(psi2)
    psi1density_history = [] 
    psi2density_history = []                                                                          
 
    psi1_exactdensity_history = []     
    psi2_exactdensity_history = []                                                                 
    times = []                                                                                                                                                                     
    error_history1 = []                                                                                 
    error_history2 = []                                                                                

    for i in range(steps):
        psi1_exact = ssfm(psi1_exact, g1, dt, k, L_half)     
        psi2_exact = ssfm(psi2_exact, g2, dt, k, L_half)      
        psi1 = filtered_qssfm(psi1, g1, dt, k, L_half, N, N_reduce1)
        psi2 = filtered_qssfm(psi2, g2, dt, k, L_half, N, N_reduce2)

        times.append((i + 1)*dt)                                                                              

        psi1density_history.append(np.abs(psi1)**2/A1**2)                                                    
        psi2density_history.append(np.abs(psi2)**2/A2**2)                                                    
        psi1_exactdensity_history.append(np.abs(psi1_exact)**2/A1**2)                                           

        error_history1.append(error(psi1, psi1_exact))                                                   
        error_history2.append(error(psi2, psi2_exact))                                             

    np.savez(f"1D-b-effect.npz",    
            times=times,
            rel_l2_err_b0p5=error_history1,
            rel_l2_err_b1=error_history2,
            ) 